# AC-MOT v12 — Final live top-3 T4 run

افتح الملف على Colab بحساب عنده T4. الوضع الافتراضي هنا هو `final_test`.
هذا الرن لا يعيد تدريب detector ولا يعيد بناء 6-bank cache. هو يشغل top-3 live systems فقط:

1. `LIVE_ADAPTIVE_NO_STABILITY`
2. `LIVE_FIXED_736`
3. `LIVE_FIXED_832`

كل frame يستخدم ONE YOLOv8n inference فقط في النظام الجاري اختباره.

**Experimental code, not guaranteed gains or journal acceptance.** Official TrackEval metric implementation uses the preserved custom class-agnostic GT filter. This is not the official VisDrone benchmark protocol. The legacy and new environments differ, so compare systems rerun inside this suite, not cross-run numbers.


In [ ]:
# 1 — Choose mode and paths
from pathlib import Path
MODE = "final_test" # "evaluate_saved", "development", "final_test"
OLD_RUN = Path('/content/drive/MyDrive/VisDrone_Results/acmot_full17_recorded_20260905_170655')
TEST_DATASET = Path('/content/drive/MyDrive/visdrone/VisDrone_Zips/VisDrone2019-MOT-test-dev/VisDrone2019-MOT-test-dev')
# Set this to your separate validation/train dataset before development mode.
DEV_DATASET = Path('/content/drive/MyDrive/visdrone/VisDrone2019-MOT-val')
OUTPUT_ROOT = Path('/content/drive/MyDrive/VisDrone_Results/ACMOT_IDS')
DEV_SEQUENCE_NAMES = [] # Empty = all sequences in DEV_DATASET. Never select by test performance.
FROZEN_CONFIG = OUTPUT_ROOT / 'development_selection' / 'frozen.json'
# Reuse these directories to resume a compatible cache/replay.
CACHE_DIR = OUTPUT_ROOT / 'development_cache'
REPLAY_DIR = OUTPUT_ROOT / 'development_replay'


In [ ]:
# 2 — Drive and dependencies (GPU only needed for new detection)
from google.colab import drive
drive.mount('/content/drive')
import subprocess,sys,os,json
from datetime import datetime
assert MODE in ['evaluate_saved','development','final_test']
WORK=Path('/content/acmot_ids');WORK.mkdir(exist_ok=True)
PY=sys.executable
packages=['numpy==2.2.6','scipy==1.15.3','pandas==2.2.3','matplotlib==3.10.3','pycocotools==2.0.10','tabulate==0.9.0']
if MODE!='evaluate_saved': packages += ['ultralytics==8.3.200','lap==0.5.12','opencv-python==4.11.0.86']
subprocess.run([PY,'-m','pip','install','-q',*packages],check=True)
TE=WORK/'TrackEval'
if not TE.exists():subprocess.run(['git','clone','https://github.com/JonathonLuiten/TrackEval.git',str(TE)],check=True)
subprocess.run(['git','-C',str(TE),'checkout','12c8791b303e0a0b50f753af204249e622d0281a'],check=True)
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
STAMP=datetime.now().strftime('%Y%m%d_%H%M%S')


In [ ]:
# 3 — Embedded source files, no separate upload needed
SOURCES = {'core.py': '"""AC-MOT experimental controls. No measured improvements are assumed."""\nfrom collections import deque\nfrom dataclasses import dataclass, asdict\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\n\nULTRALYTICS = \'8.3.200\'\nTRACKEVAL = \'12c8791b303e0a0b50f753af204249e622d0281a\'\nCLASSES = [0, 2, 5, 7]  # COCO person/car/bus/truck. No fabricated van class.\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda:f.read(2**20), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\ndef fingerprint(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()\n\ndef atomic_json(path, value):\n    path = Path(path)\n    tmp = path.with_suffix(path.suffix+\'.partial\')\n    tmp.write_text(json.dumps(value, indent=2, allow_nan=False))\n    tmp.replace(path)\n\ndef boxes(value):\n    a = np.asarray(value, dtype=np.float32).reshape(-1, 6)\n    if not np.isfinite(a).all() or np.any(a[:,2:4] <= a[:,:2]):\n        raise ValueError(\'Invalid detection coordinates\')\n    if np.any((a[:,4] < 0) | (a[:,4] > 1)) or np.any(a[:,5] != np.floor(a[:,5])):\n        raise ValueError(\'Invalid score or class\')\n    return a\n\n@dataclass(frozen=True)\nclass Config:\n    name: str\n    policy: str = \'fixed\'\n    size: int = 640\n    recovery: bool = False\n    stable: bool = False\n    detector_feedback: bool = False\n    adaptive_nms: bool = False\n    nms: float = .45\n    high: float = .25\n    low: float = .10\n    new: float = .25\n    buffer: int = 30\n    match: float = .80\n    fuse: bool = True\n    adaptive_birth: bool = False\n\n    def validate(self):\n        if self.policy not in [\'fixed\',\'adaptive\',\'cycle\'] or self.size not in [640,736,832]:\n            raise ValueError(\'Unknown policy or size\')\n        if not (0 <= self.low < self.high <= self.new <= 1) or not 0 < self.match <= 1 or self.buffer < 1:\n            raise ValueError(\'Invalid tracker thresholds\')\n        if not 0 < self.nms < 1:\n            raise ValueError(\'Invalid NMS\')\n        if not self.name or any(c not in \'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-\' for c in self.name):\n            raise ValueError(\'Unsafe system name\')\n        return self\n\ndef candidates():\n    base = Config(\'A0_pinned_default\')\n    tuned = dict(high=.18, low=.04, new=.20, buffer=45, match=.86)\n    controls = [base, Config(\'A1_tuned\', **tuned),\n        Config(\'R_recovery_only\', recovery=True, **tuned),\n        Config(\'A3_adaptive_reference\',policy=\'adaptive\', **tuned),\n        Config(\'R_stable_only\',policy=\'adaptive\',stable=True, **tuned),\n        Config(\'R_low_stable\',policy=\'adaptive\',stable=True,recovery=True, **tuned),\n        Config(\'R_feedback\',policy=\'adaptive\',stable=True,recovery=True,detector_feedback=True, **tuned),\n        Config(\'A3_v12_candidate\',policy=\'adaptive\',stable=True,recovery=True,detector_feedback=True,adaptive_birth=True, **tuned)]\n    for size in [640,736,832]:\n        controls.append(Config(f\'Fixed_{size}_recovery\',size=size,recovery=True, **tuned))\n    controls += [Config(\'Cycle_recovery\',policy=\'cycle\',recovery=True, **tuned),\n        Config(\'NMS_055\',policy=\'adaptive\',stable=True,recovery=True,detector_feedback=True,adaptive_birth=True,nms=.55, **tuned),\n        Config(\'NMS_adaptive\',policy=\'adaptive\',stable=True,recovery=True,detector_feedback=True,adaptive_birth=True,adaptive_nms=True, **tuned)]\n    for match in [.75,.80]:\n        controls.append(Config(f\'Assoc_{int(match*100)}\',policy=\'adaptive\',stable=True,recovery=True,detector_feedback=True,adaptive_birth=True, **(tuned|dict(match=match))))\n    for buffer in [30,60]:\n        controls.append(Config(f\'Buffer_{buffer}\',policy=\'adaptive\',stable=True,recovery=True,detector_feedback=True,adaptive_birth=True, **(tuned|dict(buffer=buffer))))\n    return [asdict(c.validate()) for c in controls]\n\nclass Controller:\n    def __init__(self, cfg):\n        self.cfg = cfg.validate()\n        self.history = deque(maxlen=7)\n        self.size = cfg.size\n        self.last_change = 1\n        self.sci = 0.\n        self.tiny = 0.\n        self.scene = \'clear\'\n        self.peak_count = 0\n        self.drop_age = 0\n\n    def choose(self, frame, visual, previous):\n        c = self.cfg\n        # Causal: only prior selected-resolution outputs enter the controller.\n        if frame == 1 or frame % 10 == 1:\n            previous = boxes(previous)\n            b = previous[previous[:,4] >= .18]\n            n = len(b)\n            self.tiny = float(np.mean((b[:,2]-b[:,0])*(b[:,3]-b[:,1]) < 1024)) if n else 0.\n            crowd = min(n/30,1.)\n            raw = .30*crowd + .30*self.tiny + .20*min(visual[\'edges\']/.14,1.)\n            raw += .10*(visual[\'brightness\']<80) + .05*(visual[\'blur\']<180)\n            self.history.append(raw)\n            self.sci = float(np.mean(self.history))\n            self.scene = (\'night\' if visual[\'brightness\']<80 else \'blur\' if visual[\'blur\']<180 else\n                          \'tiny\' if self.tiny>.50 else \'crowded\' if crowd>.65 or visual[\'edges\']>.13 else \'clear\')\n            # Short recovery probe when observations collapse, not perpetual high resolution.\n            dropped = self.peak_count >= 5 and n < .4*self.peak_count\n            self.drop_age = self.drop_age + 1 if dropped else 0\n            self.peak_count = max(n, int(self.peak_count*.8))\n            target = 832 if self.sci>.60 or self.tiny>.50 else 736 if self.sci>.35 or self.scene in [\'crowded\',\'tiny\'] else 640\n            if c.detector_feedback and 0 < self.drop_age <= 3:\n                target = 832\n            if c.stable:\n                if target < self.size:\n                    if self.size == 832 and (self.sci>.50 or self.tiny>.40): target=832\n                    elif self.size == 736 and self.sci>.25: target=736\n                if frame-self.last_change < 30: target=self.size\n            if c.policy == \'adaptive\' and target != self.size:\n                self.size, self.last_change = target, frame\n        size = self.size if c.policy == \'adaptive\' else [640,736,832][((frame-1)//30)%3] if c.policy == \'cycle\' else c.size\n        # NMS alternatives have their own exact cache banks, never reapplied to already suppressed boxes.\n        nms = .55 if c.adaptive_nms and self.scene in [\'crowded\',\'tiny\'] else c.nms\n        adaptive = float(np.clip(.245-.050*self.sci - (.012 if self.scene in [\'crowded\',\'tiny\',\'night\'] else 0),.19,.28))\n        conf = c.low if c.recovery else adaptive if c.policy==\'adaptive\' else .25\n        high = adaptive if c.adaptive_birth else c.high\n        new = max(high,c.new) if not c.adaptive_birth else min(1.,high+.02)\n        return dict(size=size,nms=nms,conf=conf,high=high,new=new,sci=self.sci,scene=self.scene)\n\ndef iou(a,b):\n    a,b=np.asarray(a).reshape(-1,4),np.asarray(b).reshape(-1,4)\n    inter=np.maximum(0,np.minimum(a[:,None,2:],b[None,:,2:])-np.maximum(a[:,None,:2],b[None,:,:2])).prod(2)\n    union=(a[:,2:]-a[:,:2]).prod(1)[:,None]+(b[:,2:]-b[:,:2]).prod(1)[None,:]-inter\n    return np.divide(inter,union,out=np.zeros_like(inter,dtype=float),where=union>0)\n', 'experiment.py': '"""Cache once, replay ByteTrack candidates, or measure a frozen system on T4."""\nimport argparse\nimport gzip\nimport json\nimport importlib.metadata as metadata\nimport time\nimport sys\nimport subprocess\nfrom pathlib import Path\nfrom types import SimpleNamespace\nfrom dataclasses import asdict\nimport numpy as np\nfrom core import Config,Controller,ULTRALYTICS,CLASSES,atomic_json,sha,fingerprint,boxes,candidates\n\ndef environment():\n    if metadata.version(\'ultralytics\') != ULTRALYTICS:\n        raise RuntimeError(f\'Requires ultralytics=={ULTRALYTICS}\')\n    return {p:metadata.version(p) for p in [\'ultralytics\',\'numpy\',\'torch\',\'torchvision\',\'scipy\']}\n\ndef dataset_manifest(dataset, names):\n    if not names or len(set(names))!=len(names): raise ValueError(\'Empty/duplicate sequence list\')\n    result=[]\n    for name in names:\n        if Path(name).name != name: raise ValueError(\'Invalid sequence name\')\n        files=sorted((dataset/\'sequences\'/name).glob(\'*.jpg\'))\n        if not files or [int(f.stem) for f in files]!=list(range(1,len(files)+1)):\n            raise ValueError(f\'Missing/non-contiguous frames: {name}\')\n        ann=dataset/\'annotations\'/f\'{name}.txt\'\n        gt=np.loadtxt(ann,delimiter=\',\',ndmin=2)\n        if gt.shape[1]!=10 or not np.isfinite(gt).all() or np.any(gt[:,0]<1) or np.any(gt[:,0]>len(files)):\n            raise ValueError(f\'Invalid GT: {ann}\')\n        result.append(dict(sequence=name,frames=len(files),annotation_sha256=sha(ann),\n                           frame_sha256={f.name:sha(f) for f in files}))\n    return result\n\ndef visual(img):\n    import cv2\n    gray=cv2.cvtColor(cv2.resize(img,None,fx=.25,fy=.25),cv2.COLOR_BGR2GRAY)\n    return dict(brightness=float(gray.mean()),blur=float(cv2.Laplacian(gray,cv2.CV_64F).var()),\n                edges=float(cv2.Canny(gray,50,120).mean()/255))\n\ndef sync():\n    import torch\n    torch.cuda.synchronize()\n\ndef new_model(weights):\n    import torch\n    from ultralytics import YOLO\n    if not torch.cuda.is_available() or \'T4\' not in torch.cuda.get_device_name(0):\n        raise RuntimeError(\'T4 required for detector cache and live timing\')\n    return YOLO(str(weights))\n\ndef detect(model,img,size,nms):\n    r=model.predict(img,conf=.01,iou=nms,imgsz=size,classes=CLASSES,max_det=1000,\n                    half=False,device=0,verbose=False)[0]\n    if bool(model.predictor.model.fp16): raise RuntimeError(\'Unexpected FP16\')\n    return boxes(r.boxes.data.cpu().numpy())\n\ndef cache(a):\n    import cv2\n    env=environment()\n    names=json.loads(a.sequences.read_text())\n    manifest=dataset_manifest(a.dataset,names)\n    cfg=dict(dataset=str(a.dataset),split=a.split,manifest=manifest,environment=env,\n             weights_sha256=sha(a.weights),sizes=[640,736,832],nms=[.45,.55],conf=.01,\n             max_det=1000,classes=CLASSES,precision=\'FP32\',source_sha256=sha(Path(__file__)),\n             core_sha256=sha(Path(__file__).with_name(\'core.py\')))\n    cfg[\'fingerprint\']=fingerprint(cfg)\n    a.output.mkdir(parents=True,exist_ok=True)\n    meta=a.output/\'cache.json\'\n    if meta.exists() and json.loads(meta.read_text())!=cfg: raise ValueError(\'Resume cache mismatch\')\n    atomic_json(meta,cfg)\n    model=new_model(a.weights)\n    for seq in manifest:\n        name=seq[\'sequence\']; target=a.output/f\'{name}.jsonl.gz\'; receipt=a.output/f\'{name}.complete.json\'\n        if target.exists() and receipt.exists() and json.loads(receipt.read_text())[\'sha256\']==sha(target): continue\n        first=cv2.imread(str(a.dataset/\'sequences\'/name/next(iter(seq[\'frame_sha256\']))))\n        for size in cfg[\'sizes\']: detect(model,first,size,.45)\n        partial=target.with_suffix(\'.partial\')\n        with gzip.open(partial,\'wt\') as f:\n            for index,filename in enumerate(seq[\'frame_sha256\'],1):\n                img=cv2.imread(str(a.dataset/\'sequences\'/name/filename))\n                if img is None: raise ValueError(f\'Unreadable {filename}\')\n                bank={}; times={}\n                for size in cfg[\'sizes\']:\n                    for nms in cfg[\'nms\']:\n                        key=f\'{size}_{nms:.2f}\'; sync(); start=time.perf_counter()\n                        bank[key]=detect(model,img,size,nms).tolist(); sync()\n                        times[key]=time.perf_counter()-start\n                f.write(json.dumps(dict(frame=index,shape=list(img.shape[:2]),visual=visual(img),bank=bank,cache_detection_seconds=times))+\'\\n\')\n        partial.replace(target)\n        atomic_json(receipt,dict(sha256=sha(target),fingerprint=cfg[\'fingerprint\']))\n        print(\'Cached\',name,flush=True)\n\ndef make_tracker(cfg):\n    from ultralytics.trackers.byte_tracker import BYTETracker\n    return BYTETracker(SimpleNamespace(track_high_thresh=cfg.high,track_low_thresh=cfg.low,\n        new_track_thresh=cfg.new,track_buffer=cfg.buffer,match_thresh=cfg.match,fuse_score=cfg.fuse),frame_rate=30)\n\ndef track(tracker,dets,shape,params):\n    from ultralytics.engine.results import Boxes\n    dets=boxes(dets); dets=dets[dets[:,4]>=params[\'conf\']]\n    tracker.args.track_high_thresh=params[\'high\'];tracker.args.new_track_thresh=params[\'new\']\n    result=np.asarray(tracker.update(Boxes(dets,shape)),dtype=float).reshape(-1,8)\n    if len(set(result[:,4]))!=len(result): raise ValueError(\'Duplicate track IDs\')\n    return result,dets\n\ndef recording(frame,t,params,elapsed):\n    return dict(frame=frame,ids=t[:,4].astype(int).tolist(),boxes_xyxy=t[:,:4].tolist(),\n                scores=t[:,5].tolist(),classes=t[:,6].astype(int).tolist(),settings=params,\n                elapsed_seconds=elapsed)\n\ndef replay(a):\n    env=environment(); cache_cfg=json.loads((a.cache/\'cache.json\').read_text())\n    if env!=cache_cfg[\'environment\']: raise ValueError(\'Cache/replay environment mismatch\')\n    configs=json.loads(a.config.read_text()) if a.config else candidates()\n    if cache_cfg[\'split\']!=\'development\' and not a.frozen:\n        raise ValueError(\'Search prohibited on test data. Supply a frozen development selection.\')\n    if a.frozen:\n        locked=json.loads(a.frozen.read_text())\n        if locked.get(\'development_split\')!=\'development\': raise ValueError(\'Invalid development lock\')\n        configs=locked[\'systems\']\n    checked=[Config(**c).validate() for c in configs]\n    if len({c.name for c in checked})!=len(checked):raise ValueError(\'Duplicate system names\')\n    cfg=dict(systems=[asdict(c) for c in checked],cache_fingerprint=cache_cfg[\'fingerprint\'],\n        dataset=cache_cfg[\'dataset\'],split=cache_cfg[\'split\'],environment=env,weights_sha256=cache_cfg[\'weights_sha256\'],\n        source_sha256=sha(Path(__file__)),core_sha256=sha(Path(__file__).with_name(\'core.py\')),\n        fps_protocol=\'Replay wall time only; never deployment FPS\',\n        ground_truth_filter=dict(categories=[1,4,5,6,9],score=1,occlusion_lt=2,truncation_lt=2))\n    a.output.mkdir(parents=True,exist_ok=True)\n    config_path=a.output/\'configuration.json\'\n    if config_path.exists() and json.loads(config_path.read_text())!=cfg:raise ValueError(\'Replay resume mismatch\')\n    atomic_json(config_path,cfg);atomic_json(a.output/\'dataset_manifest.json\',cache_cfg[\'manifest\'])\n    for c in checked:\n        folder=a.output/c.name;folder.mkdir(exist_ok=True)\n        for seq in cache_cfg[\'manifest\']:\n            sn=seq[\'sequence\'];target=folder/f\'{sn}.frames.jsonl.gz\';receipt=folder/f\'{sn}.complete.json\'\n            src=a.cache/f\'{sn}.jsonl.gz\'\n            cache_receipt=json.loads((a.cache/f\'{sn}.complete.json\').read_text())\n            if sha(src)!=cache_receipt[\'sha256\']:raise ValueError(\'Corrupt cache\')\n            if target.exists() and receipt.exists() and sha(target)==json.loads(receipt.read_text())[\'sha256\']:continue\n            tracker=make_tracker(c);control=Controller(c);previous=[];stats=[]; sizes=[]\n            partial=target.with_suffix(\'.partial\');count=0\n            with gzip.open(src,\'rt\') as source,gzip.open(partial,\'wt\') as dest:\n                for count,line in enumerate(source,1):\n                    r=json.loads(line)\n                    if r[\'frame\']!=count: raise ValueError(\'Missing frame\')\n                    start=time.perf_counter();params=control.choose(count,r[\'visual\'],previous)\n                    key=f"{params[\'size\']}_{params[\'nms\']:.2f}"\n                    t,d=track(tracker,r[\'bank\'][key],tuple(r[\'shape\']),params)\n                    elapsed=time.perf_counter()-start\n                    previous=d if c.detector_feedback else t[:,[0,1,2,3,5,6]]\n                    stats.append(elapsed);sizes.append(params[\'size\'])\n                    dest.write(json.dumps(recording(count,t,params,elapsed))+\'\\n\')\n            if count!=seq[\'frames\']:raise ValueError(\'Truncated cache\')\n            partial.replace(target)\n            atomic_json(receipt,dict(sha256=sha(target),replay_seconds=sum(stats),frames=count,\n                resolution_switches=int(np.count_nonzero(np.diff(sizes))),mean_size=float(np.mean(sizes))))\n        print(\'Replayed\',c.name,flush=True)\n\ndef live(a):\n    """Fresh detector + controller + tracker timing for a frozen candidate, repeated."""\n    import cv2\n    import torch\n    environment();locked=json.loads(a.frozen.read_text())\n    if locked.get(\'development_split\')!=\'development\':raise ValueError(\'Development lock required\')\n    if locked[\'weights_sha256\']!=sha(a.weights) or locked[\'environment\']!=environment():\n        raise ValueError(\'Frozen development weights/environment differ\')\n    systems=[Config(**c).validate() for c in locked[\'systems\']]\n    manifest=dataset_manifest(a.dataset,json.loads(a.sequences.read_text()))\n    a.output.mkdir(parents=True,exist_ok=False)\n    atomic_json(a.output/\'dataset_manifest.json\',manifest)\n    atomic_json(a.output/\'configuration.json\',dict(systems=[asdict(c) for c in systems],\n        weights_sha256=sha(a.weights),environment=environment(),split=a.split,\n        source_sha256=sha(Path(__file__)),core_sha256=sha(Path(__file__).with_name(\'core.py\')),\n        ground_truth_filter=dict(categories=[1,4,5,6,9],score=1,occlusion_lt=2,truncation_lt=2)))\n    measurements=[]\n    total_frames = sum(seq[\'frames\'] for seq in manifest)\n    grand_total = total_frames * len(systems) * a.repeats\n    grand_done = 0\n    grand_start = time.perf_counter()\n    print(f\'LIVE RUN START | systems={len(systems)} repeats={a.repeats} sequences={len(manifest)} frames/repeat/system={total_frames} total_frame_runs={grand_total}\', flush=True)\n    for repeat in range(a.repeats):\n        for c in systems:\n            system_start = time.perf_counter()\n            print(f\'BEGIN repeat={repeat+1}/{a.repeats} system={c.name}\', flush=True)\n            model=new_model(a.weights);total=0.;n=0;latencies=[]\n            torch.cuda.reset_peak_memory_stats()\n            for seq in manifest:\n                sn=seq[\'sequence\'];folder=a.output/f\'repeat_{repeat}\'/c.name;folder.mkdir(parents=True,exist_ok=True)\n                seq_start = time.perf_counter()\n                print(f\'  SEQ {sn} frames={seq["frames"]}\', flush=True)\n                paths=[a.dataset/\'sequences\'/sn/f for f in seq[\'frame_sha256\']]\n                first=cv2.imread(str(paths[0]))\n                for size in [640,736,832]:\n                    for _ in range(3):detect(model,first,size,c.nms)\n                control=Controller(c);tracker=make_tracker(c);previous=[]\n                with gzip.open(folder/f\'{sn}.frames.jsonl.gz\',\'wt\') as f:\n                    for frame,path in enumerate(paths,1):\n                        sync();start=time.perf_counter();img=cv2.imread(str(path))\n                        if img is None:raise ValueError(str(path))\n                        # Visual analysis follows the same schedule as replay.\n                        v=visual(img) if frame==1 or frame%10==1 else {}\n                        params=control.choose(frame,v,previous)\n                        dets=detect(model,img,params[\'size\'],params[\'nms\'])\n                        t,d=track(tracker,dets,img.shape[:2],params)\n                        previous=d if c.detector_feedback else t[:,[0,1,2,3,5,6]]\n                        sync();elapsed=time.perf_counter()-start;total+=elapsed;n+=1;latencies.append(elapsed)\n                        f.write(json.dumps(recording(frame,t,params,elapsed))+\'\\n\')\n                        grand_done += 1\n                        if frame == 1 or frame == seq[\'frames\'] or frame % 50 == 0:\n                            now = time.perf_counter()\n                            sys_elapsed = now - system_start\n                            grand_elapsed = now - grand_start\n                            fps = n / total if total > 0 else 0.0\n                            done_pct = 100.0 * grand_done / grand_total\n                            eta = (grand_elapsed / grand_done) * (grand_total - grand_done) if grand_done else 0.0\n                            print(\n                                f\'  progress={done_pct:6.2f}% global={grand_done}/{grand_total} \'\n                                f\'repeat={repeat+1}/{a.repeats} system={c.name} seq={sn} frame={frame}/{seq["frames"]} \'\n                                f\'system_elapsed={sys_elapsed/60:.1f}m ETA={eta/60:.1f}m current_FPS={fps:.2f}\',\n                                flush=True\n                            )\n                print(f\'  DONE SEQ {sn} elapsed={(time.perf_counter()-seq_start)/60:.1f}m\', flush=True)\n            measurements.append(dict(system=c.name,repeat=repeat,frames=n,seconds=total,fps=n/total,\n                p95_ms=float(np.percentile(latencies,95)*1000),peak_gpu_bytes=torch.cuda.max_memory_allocated(),\n                excludes=\'warmup and output serialization; includes frame read, analysis, detector and tracker\'))\n            atomic_json(a.output/\'timing.json\',measurements)\n            print(f\'DONE repeat={repeat+1}/{a.repeats} system={c.name} frames={n} seconds={total:.3f} fps={n/total:.3f}\', flush=True)\n            del model;torch.cuda.empty_cache()\n\ndef main():\n    p=argparse.ArgumentParser();sub=p.add_subparsers(dest=\'command\',required=True)\n    for name in [\'cache\',\'live\']:\n        s=sub.add_parser(name);s.add_argument(\'--dataset\',required=True,type=Path);s.add_argument(\'--sequences\',required=True,type=Path)\n        s.add_argument(\'--weights\',required=True,type=Path);s.add_argument(\'--output\',required=True,type=Path)\n        s.add_argument(\'--split\',required=True,choices=[\'development\',\'test\'])\n        if name==\'live\':s.add_argument(\'--frozen\',required=True,type=Path);s.add_argument(\'--repeats\',type=int,default=3)\n    s=sub.add_parser(\'replay\');s.add_argument(\'--cache\',required=True,type=Path);s.add_argument(\'--output\',required=True,type=Path)\n    s.add_argument(\'--config\',type=Path);s.add_argument(\'--frozen\',type=Path)\n    a=p.parse_args()\n    if a.command==\'live\' and a.repeats<1:raise ValueError(\'repeats must be positive\')\n    globals()[a.command](a)\n\nif __name__==\'__main__\':main()\n', 'report.py': '"""Retain every candidate and freeze only development-qualified configurations."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport numpy as np\nfrom core import atomic_json,sha\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--run\',type=Path,required=True)\n    p.add_argument(\'--evaluation\',type=Path,required=True);p.add_argument(\'--output\',type=Path,required=True)\n    a=p.parse_args();cfg=json.loads((a.run/\'configuration.json\').read_text())\n    data=json.loads((a.evaluation/\'metrics.json\').read_text())\n    baseline=\'A0_pinned_default\';reference=\'A3_adaptive_reference\'\n    if baseline not in data or reference not in data: raise ValueError(\'Both control systems are required\')\n    a.output.mkdir(parents=True,exist_ok=False)\n    def scores(r):\n        return dict(HOTA=float(np.mean(r[\'HOTA\'][\'combined\'][\'HOTA\'])),\n                    MOTA=float(r[\'CLEAR\'][\'combined\'][\'MOTA\']),IDS=int(r[\'CLEAR\'][\'combined\'][\'IDSW\']))\n    bs=scores(data[baseline]);rs=scores(data[reference]);rows=[]\n    rng=np.random.default_rng(20260906)\n    for name,r in data.items():\n        sc=scores(r);seqs=sorted(r[\'CLEAR\'][\'per_sequence\'])\n        # Cluster clips with the same UAV source prefix to avoid treating them as independent videos.\n        groups={s.split(\'_\')[0] for s in seqs}\n        clusters=[np.array([float(r[\'CLEAR\'][\'per_sequence\'][s][\'MOTA\'])-\n                    float(data[baseline][\'CLEAR\'][\'per_sequence\'][s][\'MOTA\']) for s in seqs if s.split(\'_\')[0]==g]) for g in sorted(groups)]\n        samples=[float(np.concatenate([clusters[i] for i in rng.integers(0,len(clusters),len(clusters))]).mean()) for _ in range(2000)]\n        qualified=all(sc[\'MOTA\']>=v[\'MOTA\'] and sc[\'IDS\']<=v[\'IDS\'] and sc[\'HOTA\']>=v[\'HOTA\'] for v in [bs,rs])\n        rows.append(dict(system=name,**sc,qualified=qualified,\n            macro_MOTA_delta_ci95=np.percentile(samples,[2.5,97.5]).tolist(),\n            uncertainty_scope=\'Exploratory paired source-cluster bootstrap of macro MOTA delta vs A0; not combined-score CI or selection-adjusted\'))\n    atomic_json(a.output/\'all_candidates.json\',rows)\n    eligible=[r for r in rows if r[\'qualified\'] and r[\'system\'] not in [baseline,reference]]\n    if cfg[\'split\']==\'development\' and eligible:\n        winner=max(eligible,key=lambda r:(r[\'HOTA\'],r[\'MOTA\'],-r[\'IDS\']))[\'system\']\n        selected=[c for c in cfg[\'systems\'] if c[\'name\'] in [baseline,reference,winner]]\n        atomic_json(a.output/\'frozen.json\',dict(development_split=\'development\',systems=selected,\n            weights_sha256=cfg[\'weights_sha256\'],environment=cfg[\'environment\'],\n            selection_rule=\'HOTA maximum subject to MOTA/HOTA not lower and IDS not higher than both A0 and reference A3\',\n            winner=winner,evaluation_sha256=sha(a.evaluation/\'metrics.json\'),\n            warning=\'Development selection, not evidence of held-out improvement\'))\n        print(\'Development candidate frozen:\',winner)\n    else:\n        print(\'No candidate frozen. No jointly qualifying candidate, or this is test data.\')\n\nif __name__==\'__main__\':main()\n', 'evaluate.py': '"""Official TrackEval metrics on preserved, class-agnostic research-filter tracks.\n\nThis adapter is NOT the official VisDrone benchmark preprocessing protocol.\n"""\nimport argparse\nimport gzip\nimport hashlib\nimport json\nimport subprocess\nimport sys\nimport importlib.metadata\nfrom pathlib import Path\nimport numpy as np\n\nREVISION = \'12c8791b303e0a0b50f753af204249e622d0281a\'\n\n# Pinned upstream uses removed NumPy scalar aliases. Restore aliases only;\n# no metric equations or matching logic are changed.\nfor alias, scalar in [(\'float\', float), (\'int\', int)]:\n    if alias not in np.__dict__:\n        setattr(np, alias, scalar)\n\ndef iou(a, b):\n    a = np.asarray(a, dtype=float).reshape(-1, 4)\n    b = np.asarray(b, dtype=float).reshape(-1, 4)\n    for boxes in (a, b):\n        if not np.isfinite(boxes).all() or np.any(boxes[:, 2:] <= boxes[:, :2]):\n            raise ValueError(\'Non-finite or non-positive box\')\n    inter = np.maximum(0, np.minimum(a[:, None, 2:], b[None, :, 2:]) -\n                       np.maximum(a[:, None, :2], b[None, :, :2])).prod(axis=2)\n    union = (a[:, 2:] - a[:, :2]).prod(axis=1)[:, None] + (b[:, 2:] - b[:, :2]).prod(axis=1)[None, :] - inter\n    return np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)\n\ndef prepare(gt, frames, count):\n    if [r[\'frame\'] for r in frames] != list(range(1, count + 1)):\n        raise ValueError(\'Recording must contain every frame exactly once, in order\')\n    gt = np.asarray(gt, dtype=float).reshape(-1, 10)\n    if not np.isfinite(gt).all() or np.any(gt[:, 0] < 1) or np.any(gt[:, 0] > count):\n        raise ValueError(\'Invalid annotation frame or value\')\n    gt = gt[np.isin(gt[:, 7], [1, 4, 5, 6, 9]) & (gt[:, 6] == 1) & (gt[:, 8] < 2) & (gt[:, 9] < 2)]\n    gids = sorted(set(gt[:, 1].tolist()))\n    pids = sorted({v for r in frames for v in r[\'ids\']})\n    gm, pm = {v:i for i,v in enumerate(gids)}, {v:i for i,v in enumerate(pids)}\n    data = dict(num_timesteps=count, num_gt_ids=len(gids), num_tracker_ids=len(pids),\n                num_gt_dets=len(gt), num_tracker_dets=sum(len(r[\'ids\']) for r in frames),\n                gt_ids=[], tracker_ids=[], similarity_scores=[])\n    for r in frames:\n        g = gt[gt[:, 0] == r[\'frame\']]\n        ids = r[\'ids\']\n        if len(ids) != len(r[\'boxes_xyxy\']) or len(set(ids)) != len(ids) or len(set(g[:, 1])) != len(g):\n            raise ValueError(\'Duplicate IDs or mismatched boxes/IDs\')\n        gb = g[:, 2:6].copy()\n        gb[:, 2:] += gb[:, :2]\n        data[\'gt_ids\'].append(np.array([gm[v] for v in g[:, 1]], dtype=int))\n        data[\'tracker_ids\'].append(np.array([pm[v] for v in ids], dtype=int))\n        data[\'similarity_scores\'].append(iou(gb, r[\'boxes_xyxy\']))\n    return data\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\'run\', type=Path)\n    p.add_argument(\'--dataset\', required=True, type=Path)\n    p.add_argument(\'--trackeval\', required=True, type=Path)\n    p.add_argument(\'--output\', required=True, type=Path)\n    a = p.parse_args()\n    rev = subprocess.check_output([\'git\', \'-C\', str(a.trackeval), \'rev-parse\', \'HEAD\'], text=True).strip()\n    if rev != REVISION:\n        raise ValueError(\'TrackEval revision differs from pinned revision\')\n    if subprocess.check_output([\'git\',\'-C\',str(a.trackeval),\'status\',\'--porcelain\',\'--untracked-files=no\'],text=True).strip():\n        raise ValueError(\'TrackEval has modified tracked files\')\n    sys.path.insert(0, str(a.trackeval))\n    import trackeval\n    metrics = [trackeval.metrics.HOTA(), trackeval.metrics.CLEAR({\'THRESHOLD\': .5, \'PRINT_CONFIG\': False}),\n               trackeval.metrics.Identity({\'THRESHOLD\': .5, \'PRINT_CONFIG\': False})]\n    cfg = json.loads((a.run/\'configuration.json\').read_text())\n    manifest = json.loads((a.run/\'dataset_manifest.json\').read_text())\n    if not manifest or len({s[\'sequence\'] for s in manifest}) != len(manifest):\n        raise ValueError(\'Empty or duplicate sequence manifest\')\n    results, hashes = {}, {}\n    for system in cfg[\'systems\']:\n        name = system[\'name\']\n        per_metric = {m.get_name(): {} for m in metrics}\n        for seq in manifest:\n            sn = seq[\'sequence\']\n            ann = a.dataset/\'annotations\'/f\'{sn}.txt\'\n            digest = hashlib.sha256(ann.read_bytes()).hexdigest()\n            if digest != seq[\'annotation_sha256\']:\n                raise ValueError(f\'Annotation hash differs: {sn}\')\n            recording = a.run/name/f\'{sn}.frames.jsonl.gz\'\n            hashes[str(recording)] = hashlib.sha256(recording.read_bytes()).hexdigest()\n            with gzip.open(recording, \'rt\') as f:\n                frames = [json.loads(line) for line in f]\n            data = prepare(np.loadtxt(ann, delimiter=\',\', ndmin=2), frames, seq[\'frames\'])\n            for m in metrics:\n                per_metric[m.get_name()][sn] = m.eval_sequence(data)\n        results[name] = {m.get_name(): dict(per_sequence=per_metric[m.get_name()],\n            combined=m.combine_sequences(per_metric[m.get_name()])) for m in metrics}\n    a.output.mkdir(parents=True, exist_ok=False)\n    def serial(v):\n        return v.tolist() if hasattr(v, \'tolist\') else v\n    (a.output/\'metrics.json\').write_text(json.dumps(results, default=serial, indent=2, allow_nan=False))\n    import csv\n    with (a.output/\'summary.csv\').open(\'w\') as f:\n        writer = csv.writer(f)\n        writer.writerow([\'system\', \'HOTA\', \'DetA\', \'AssA\', \'MOTA\', \'IDF1\', \'IDS\', \'FN\', \'FP\'])\n        for name, r in results.items():\n            h, c, i = [r[k][\'combined\'] for k in [\'HOTA\', \'CLEAR\', \'Identity\']]\n            writer.writerow([name, *[float(np.mean(h[k])) for k in [\'HOTA\',\'DetA\',\'AssA\']],\n                             c[\'MOTA\'], i[\'IDF1\'], c[\'IDSW\'], c[\'CLR_FN\'], c[\'CLR_FP\']])\n    (a.output/\'protocol.json\').write_text(json.dumps(dict(trackeval_commit=rev,\n        protocol=\'Custom class-agnostic research filter; no benchmark ignore-region preprocessing\',\n        official_visdrone=False, gt_filter=cfg[\'ground_truth_filter\'],\n        aggregation=\'TrackEval combine_sequences, not macro mean\', scale=\'0–1\',\n        packages={p:importlib.metadata.version(p) for p in [\'numpy\',\'scipy\']},\n        source_hashes=hashes, evaluator_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()), indent=2))\n    print(a.output/\'summary.csv\')\n\nif __name__ == \'__main__\':\n    main()\n'}
for name,source in SOURCES.items():
    compile(source,name,'exec')
    (WORK/name).write_text(source)
def run(script,*args):
    cmd=[PY,str(WORK/script),*map(str,args)]
    result=subprocess.run(cmd,cwd=WORK,text=True,capture_output=True)
    if result.stdout: print(result.stdout)
    if result.stderr: print(result.stderr,file=sys.stderr)
    result.check_returncode()
with (OUTPUT_ROOT/f'environment_{STAMP}.txt').open('w') as f:
    subprocess.run([PY,'-m','pip','freeze'],stdout=f,check=True)


In [ ]:
# 4 — Run the selected workflow
if MODE=='evaluate_saved':
    assert (OLD_RUN/'configuration.json').exists(),f'Missing recorded run: {OLD_RUN}'
    EVALUATION=OUTPUT_ROOT/f'old_recording_trackeval_{STAMP}'
    run('evaluate.py',OLD_RUN,'--dataset',TEST_DATASET,'--trackeval',TE,'--output',EVALUATION)
elif MODE=='development':
    assert 'test' not in DEV_DATASET.name.lower(),'Use a separate development split, not test-dev.'
    assert (DEV_DATASET/'sequences').exists(),f'Set DEV_DATASET to an existing validation/train dataset: {DEV_DATASET}'
    names=DEV_SEQUENCE_NAMES or sorted(p.name for p in (DEV_DATASET/'sequences').iterdir() if p.is_dir())
    sequence_file=WORK/'sequences.json';sequence_file.write_text(json.dumps(names))
    # Weights download happens once; exact bytes are hashed into the cache manifest.
    subprocess.run([PY,'-c',"from ultralytics import YOLO; YOLO('yolov8n.pt')"],cwd=WORK,check=True)
    weights=WORK/'yolov8n.pt'
    run('experiment.py','cache','--dataset',DEV_DATASET,'--sequences',sequence_file,'--weights',weights,'--split','development','--output',CACHE_DIR)
    run('experiment.py','replay','--cache',CACHE_DIR,'--output',REPLAY_DIR)
    EVALUATION=OUTPUT_ROOT/f'development_trackeval_{STAMP}'
    run('evaluate.py',REPLAY_DIR,'--dataset',DEV_DATASET,'--trackeval',TE,'--output',EVALUATION)
    SELECTION=OUTPUT_ROOT/f'development_selection_{STAMP}'
    run('report.py','--run',REPLAY_DIR,'--evaluation',EVALUATION,'--output',SELECTION)
    print('If a candidate qualifies, use this path for final_test:',SELECTION/'frozen.json')
else:
    import importlib.metadata as metadata
    names=sorted(p.name for p in (TEST_DATASET/'sequences').iterdir() if p.is_dir())
    assert len(names)==17,f'Expected 17 test-dev sequences, found {len(names)}'
    frame_count=sum(len(list((TEST_DATASET/'sequences'/n).glob('*.jpg'))) for n in names)
    assert frame_count==6635,f'Expected 6635 frames, found {frame_count}'
    sequence_file=WORK/'sequences.json';sequence_file.write_text(json.dumps(names))
    subprocess.run([PY,'-c',"from ultralytics import YOLO; YOLO('yolov8n.pt')"],cwd=WORK,check=True)
    from core import sha, atomic_json
    LIVE_TOP3 = [
        dict(name='LIVE_ADAPTIVE_NO_STABILITY', policy='adaptive', size=640,
             recovery=True, stable=False, detector_feedback=True, adaptive_nms=False,
             adaptive_birth=True, nms=0.45, high=0.18, low=0.04, new=0.20,
             buffer=45, match=0.86, fuse=True),
        dict(name='LIVE_FIXED_736', policy='fixed', size=736,
             recovery=True, stable=False, detector_feedback=False, adaptive_nms=False,
             adaptive_birth=False, nms=0.45, high=0.18, low=0.04, new=0.20,
             buffer=45, match=0.86, fuse=True),
        dict(name='LIVE_FIXED_832', policy='fixed', size=832,
             recovery=True, stable=False, detector_feedback=False, adaptive_nms=False,
             adaptive_birth=False, nms=0.45, high=0.18, low=0.04, new=0.20,
             buffer=45, match=0.86, fuse=True),
    ]
    FROZEN_CONFIG = WORK/'live_top3_frozen.json'
    atomic_json(FROZEN_CONFIG, dict(
        development_split='development',
        systems=LIVE_TOP3,
        weights_sha256=sha(WORK/'yolov8n.pt'),
        environment={p:metadata.version(p) for p in ['ultralytics','numpy','torch','torchvision','scipy']},
        selection_rule='Top-3 carried forward from saved Round 2: strongest adaptive, fixed 736 middle baseline, fixed 832 quality upper baseline',
        warning='Exploratory test-dev live comparison; not a held-out final benchmark claim'
    ))
    print('GPU:', __import__('torch').cuda.get_device_name(0))
    print('Sequences:', len(names), '| frames:', frame_count)
    print('Frozen top-3 config:', FROZEN_CONFIG)
    LIVE=OUTPUT_ROOT/f'final_live_{STAMP}'
    run('experiment.py','live','--dataset',TEST_DATASET,'--sequences',sequence_file,'--weights',WORK/'yolov8n.pt','--split','test','--frozen',FROZEN_CONFIG,'--repeats',3,'--output',LIVE)
    import shutil
    # Evaluate every timing repeat independently, keeping each source recording.
    for repeat in range(3):
        repeat_run=LIVE/f'repeat_{repeat}'
        for name in ['configuration.json','dataset_manifest.json']:shutil.copy2(LIVE/name,repeat_run/name)
        EVALUATION=LIVE/f'trackeval_repeat_{repeat}'
        run('evaluate.py',repeat_run,'--dataset',TEST_DATASET,'--trackeval',TE,'--output',EVALUATION)
    print('All repeat results and synchronized timing:',LIVE)


In [ ]:
# 5 — Show and download results (all scores use 0–1 scale)
import csv
from IPython.display import display,HTML
import html
rows=list(csv.reader((EVALUATION/'summary.csv').open()))
display(HTML('<table>'+''.join('<tr>'+''.join('<td>'+html.escape(v)+'</td>' for v in row)+'</tr>' for row in rows)+'</table>'))
print('Saved on Drive:',EVALUATION)
print('HOTA: official implementation. Dataset protocol: custom class-agnostic research filter.')
# Optional download:
# from google.colab import files
# files.download(str(EVALUATION/'summary.csv'))


## Interpretation and remaining publication requirements

- No output is automatically promoted to a final result. All candidates remain recorded, even if they lose.
- Confidence recovery, stability, feedback, adaptive track birth, NMS, matching and buffer controls are separate candidates. A4 deep ReID is not implemented or claimed.
- The multiresolution/NMS cache costs six detector passes per frame once, then enables CPU tracker replay. Cached detections are post-NMS. New NMS values or lower confidence than 0.01 require a new bank.
- Dataset frame hashes and model hashes protect resume. Replay timing is not deployment FPS. final_test measures synchronized T4 FP32 execution, 3 repeats, warmup excluded, image reading included.
- COCO person/car/bus/truck are pooled for the custom five-GT-class evaluation. There is no exact COCO van class. Official class-wise VisDrone preprocessing and ignored-region evaluation still require a separately validated benchmark adapter. Do not claim official VisDrone scores.
- The development bootstrap is exploratory. For publication use untouched data, matched external trackers, additional dataset evidence, and a contribution beyond parameter tuning.
